# 01 — Explore Eulerian Fields

This notebook demonstrates how to load and visualize the **Eulerian velocity and pressure fields** from the CylinderWake3900 dataset.

**Dataset**: DNS of flow past a smooth cylinder at Re = 3900 (Incompact3d)  
**Paper**: [Khojasteh et al. (2021), Data in Brief](https://doi.org/10.1016/j.dib.2021.107725)  
**Data**: [doi:10.15454/GLNRHK](https://doi.org/10.15454/GLNRHK)

In [ ]:
# Install if needed
# !pip install cylinderwake3900[all]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cylinderwake import CylinderWake3900
from cylinderwake.visualize import plot_velocity_field, plot_vorticity, compute_vorticity

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load the Dataset

The `CylinderWake3900` class auto-downloads data from INRAE and converts to HDF5 on first use.

In [ ]:
# Load near-wake Eulerian data
ds = CylinderWake3900(field='eulerian', subdomain='near', download=True)

print(f'Number of snapshots: {len(ds)}')
print(f'Dataset metadata:')
print(ds.to_json())

## 2. Inspect a Single Snapshot

In [ ]:
sample = ds[0]

print(f"Velocity shape: {sample['velocity'].shape}")    # (3, Nx, Ny, Nz)
print(f"Pressure shape: {sample['pressure'].shape}")    # (1, Nx, Ny, Nz)
print(f"Time: {sample['time']}")

velocity = sample['velocity'].numpy() if hasattr(sample['velocity'], 'numpy') else sample['velocity']
print(f"\nVelocity statistics:")
print(f"  ux: min={velocity[0].min():.4f}, max={velocity[0].max():.4f}, mean={velocity[0].mean():.4f}")
print(f"  uy: min={velocity[1].min():.4f}, max={velocity[1].max():.4f}, mean={velocity[1].mean():.4f}")
print(f"  uz: min={velocity[2].min():.4f}, max={velocity[2].max():.4f}, mean={velocity[2].mean():.4f}")

## 3. Visualize Velocity Field (Mid-plane Slice)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (comp, label) in enumerate(zip(['ux', 'uy', 'uz'], ['$u_x$', '$u_y$', '$u_z$'])):
    plot_velocity_field(
        velocity, grid=sample.get('grid'),
        component=comp, slice_axis='z',
        ax=axes[i], title=f'{label} (z mid-plane)',
    )

plt.tight_layout()
plt.savefig('eulerian_velocity_components.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Velocity Magnitude

In [ ]:
fig = plot_velocity_field(
    velocity, grid=sample.get('grid'),
    component='magnitude', slice_axis='z',
    cmap='magma', title='Velocity magnitude |u| (z mid-plane)',
)
plt.savefig('velocity_magnitude.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Vorticity Field

The z-component of vorticity reveals the classic von Kármán vortex street.

In [ ]:
omega = compute_vorticity(velocity, grid=sample.get('grid'))
print(f'Vorticity shape: {omega.shape}')

fig = plot_vorticity(
    velocity, grid=sample.get('grid'),
    component='z', slice_axis='z',
    cmap='RdBu_r',
)
plt.savefig('vorticity_z.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Temporal Sequence

Load a contiguous sequence for forecasting tasks.

In [ ]:
seq = ds.get_sequence(start=0, length=5)

print(f"Sequence velocity shape: {seq['velocity'].shape}")  # (5, 3, Nx, Ny, Nz)
print(f"Times: {seq['times']}")

# Visualize sequence
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
vel_np = seq['velocity'].numpy() if hasattr(seq['velocity'], 'numpy') else seq['velocity']

for t in range(5):
    mag = np.sqrt(vel_np[t, 0]**2 + vel_np[t, 1]**2 + vel_np[t, 2]**2)
    nz = mag.shape[2]
    axes[t].imshow(mag[:, :, nz//2].T, origin='lower', cmap='magma', aspect='auto')
    axes[t].set_title(f't = {seq["times"][t]:.1f}')
    axes[t].axis('off')

plt.suptitle('Velocity magnitude evolution (z mid-plane)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Using with PyTorch DataLoader

In [ ]:
from torch.utils.data import DataLoader

train_ds = CylinderWake3900('eulerian', 'near', split='train')
val_ds   = CylinderWake3900('eulerian', 'near', split='val')
test_ds  = CylinderWake3900('eulerian', 'near', split='test')

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)

for batch in train_loader:
    print(f"Batch velocity shape: {batch['velocity'].shape}")
    print(f"Batch pressure shape: {batch['pressure'].shape}")
    break

---

**Next**: See `02_explore_lagrangian.ipynb` for particle trajectory analysis, and `03_baseline_forecasting.ipynb` for a simple ML baseline.